# King County: Begleitnotizen auf Deutsch

Nachschlage-Notebook neben `02_king_county_project_dominic.ipynb`: Begriffe, Begründungen und Befunde auf Deutsch, in der Reihenfolge der Projektschritte. Kein Code, nur zum Lesen. Wird mit dem Projekt fortgeschrieben.

## Die Features auf Deutsch

**Umrechnung:** 1 sqft (square foot) ≈ 0,093 m², Faustregel: **sqft ÷ 10,8 = m²** (2.000 sqft ≈ 186 m²)

| Spalte | Deutsch | Beispiel / Hinweis |
|:--|:--|:--|
| `sqft_living` | **Wohnfläche** des Hauses (alle Stockwerke inkl. Keller) | 2.000 sqft ≈ 186 m² |
| `sqft_above` | Wohnfläche **über der Erde** (ohne Keller) | |
| `sqft_basement` | Fläche des **Kellers** | es gilt immer: `living = above + basement` |
| `sqft_lot` | Fläche des **Grundstücks** (Parzelle, Haus + Garten) | 5.000 sqft ≈ 465 m² |
| `sqft_living15` | durchschnittliche Wohnfläche der **15 nächsten Nachbarhäuser** | „15“ = 15 Nachbarn; beschreibt die Nachbarschaft |
| `sqft_lot15` | durchschnittliche **Grundstücksfläche** der 15 nächsten Nachbarn | |
| `bedrooms` | Anzahl **Schlafzimmer** | |
| `bathrooms` | Anzahl **Bäder** (in Bruchteilen) | 0,5 = nur WC + Waschbecken, 0,75 = mit Dusche, 1 = mit Wanne |
| `floors` | Anzahl **Stockwerke** | 1,5 = mit ausgebautem Dachgeschoss |
| `waterfront` | Lage **direkt am Wasser** | 1 = ja, 0 = nein |
| `view` | Qualität der **Aussicht** | 0 = keine … 4 = hervorragend |
| `condition` | **Zustand** des Hauses | 1 = schlecht … 5 = sehr gut |
| `grade` | **Bau- und Ausstattungsqualität** (Bewertungssystem von King County) | 1–13; 7 = Durchschnitt, 11–13 = Luxus |
| `yr_built` | **Baujahr** | |
| `yr_renovated` | Jahr der **Renovierung** | 0 = nie renoviert |
| `zipcode` | **Postleitzahl** | Kategorie, keine Zahl |
| `lat` / `long` | **Breiten- / Längengrad** | `lat` größer = weiter nördlich |
| `date` | **Verkaufsdatum** | |
| `price` | **Verkaufspreis** in $ (Zielgröße) | |

## Notebook 01: Zusammenführen der Tabellen

- **Schlüssel:** `house_id` in `sales` ↔ `id` in `details`. Achtung: `sales` hat auch eine Spalte `id`, das ist die **Verkaufsnummer**, nicht die Hausnummer. Darum `suffixes=("", "_details")`.
- **Beziehung:** viele zu eins. 21.597 Verkäufe, 21.420 Häuser; 176 Häuser wurden mehrmals verkauft (bis zu 3-mal).
- **`how="left"` gegen `how="inner"`:** `left` behält jeden Verkauf (fehlt das Haus, stehen `NaN`), `inner` behält nur Schlüssel aus beiden Tabellen und lässt Verkäufe ohne Haus **still** verschwinden. Hier fehlt nichts (0 und 0), also liefern alle Arten dasselbe; `left` ist die Absicherung.
- **`eda.csv`** wird erzeugt, nicht in git gespeichert: erst Notebook 01 ausführen.

## Schritt 1: Die Metriken

Für jedes Haus ist der Fehler (Residuum) **eᵢ = yᵢ − ŷᵢ** (wahrer Preis minus Vorhersage).

| | Formel | Einheit | in Worten | perfekt |
|:--|:--|:--|:--|:--|
| **MAE** | (1/n) · Σ \|eᵢ\| | $ | „Im Schnitt liegt das Modell um X $ daneben.“ | 0 |
| **RMSE** | √( (1/n) · Σ eᵢ² ) | $ | „Typischer Fehler, große Fehler zählen überproportional.“ | 0 |
| **R²** | 1 − Σ eᵢ² / Σ (yᵢ − ȳ)² | ohne | „Anteil der Preisstreuung, den das Modell erklärt.“ | 1 |

- **MAE gegen RMSE:** MAE zählt jeden Dollar gleich, RMSE quadriert zuerst (doppelter Fehler = vierfaches Gewicht). Immer gilt RMSE ≥ MAE. Bei „normalen“ Fehlern ist RMSE ≈ 1,25 × MAE; liegt das Verhältnis viel höher, gibt es **wenige sehr große Fehler**.
- **RMSE gegen R²:** auf denselben Daten dieselbe Rangfolge (R² = 1 − RMSE² / Var(y)). R² ist nur die Übersetzung „in Prozent erklärt“.
- **Mittelwert-Baseline:** R² = 0, RMSE = Standardabweichung von `price` (Training ≈ 372.000 $).
- **Auf dem Testset** kann R² negativ werden: schlechter als „immer den Mittelwert raten“.
- **Wie man die Metrik wählt:** nicht nach den Daten, sondern danach, **was ein Fehler den Nutzer kostet**. Unsere Annahme: eine **Bank**, die Kredite nach dem geschätzten Hauswert vergibt, für sie ist eine um 500.000 $ überschätzte Villa viel teurer als viele kleine Fehler → **RMSE**. MAE und R² stehen daneben.
- **Ausreißer ≠ großer Fehler:** ein Ausreißer ist ein ungewöhnlicher **Datenwert**, ein großer Fehler eine schlechte **Vorhersage**.

## Schritt 3: Split nach Haus

- 176 Häuser wurden mehrmals verkauft. Ein normaler `train_test_split` über die Zeilen hätte 63 Häuser auf beide Seiten verteilt (1,5 % der Testverkäufe): mildes **Leakage**, das Modell wäre an Häusern getestet worden, die es mit Preis schon kennt.
- Darum: `train_test_split` **einmal**, aber auf die Liste der `house_id`; jeder Verkauf geht auf die Seite seines Hauses. Ergebnis: 17.270 Trainings- und 4.327 Testverkäufe (20,0 %).
- **Konsequenz für die CV:** auch dort `GroupKFold` mit `house_id` als Gruppe, sonst wäre der Test sauber, die Modellwahl aber nicht. `house_id` vorher als `groups_train` sichern.

## Schritt 4: Was die EDA gezeigt hat

| Befund | Zahl | Folge |
|:--|:--|:--|
| Fehlwerte | `yr_renovated` 17,9 %, `waterfront` 11,1 %, `sqft_basement` 2,0 %, `view` 0,3 % | Preis fast gleich mit/ohne Wert → Lücken wirken zufällig (MCAR) |
| `sqft_basement` exakt füllbar | `living = above + basement` in 100 % der Zeilen | Regel in `prepare()` |
| Preis rechtsschief | Median 450.000 $, Mittelwert 542.521 $, max. 7,7 Mio. $, Schiefe 4,2 (log: 0,44) | große Fehler bei teuren Häusern erwartet; log-Preis als Modellvariante |
| stärkste Spalte | `sqft_living` (Pearson 0,70), dann `grade` (0,67) | Regel-Baseline aus `sqft_living` |
| `zipcode` | Korrelation ≈ 0, aber höchste Mutual Information (0,41); Medianpreis je PLZ 230.000 – 1.915.000 $ | Etiketten ohne Ordnung → One-Hot |
| Größen-Cluster | `sqft_above`, `bathrooms`, `grade`, `sqft_living15` hängen stark an `sqft_living` (0,76 – 0,88) | Ridge verkraftet das; Koeffizienten nicht einzeln deuten |
| `grade` | Preis steigt **beschleunigt** je Stufe | Polynomfeatures oder log-Preis in Schritt 7 |
| `waterfront` | Median ≈ 1,4 Mio. $ gegen 450.000 $, aber nur 123 Häuser | größte Fehler dort erwartet |
| Schiefe Features | `sqft_lot` 13,0, `sqft_lot15` 9,9 | `log1p` in der Pipeline |
| VIF | `living`/`above`/`basement` = ∞ (exakte Summe) | `sqft_above` löschen (danach max. VIF 6,1) |
| Plausibilität | 5 Häuser **nach** dem Verkauf renoviert, 11 vor Fertigstellung verkauft | `renovated` nur bis zum Verkaufsjahr; Alter mindestens 0 |

**Begriffe:**
- **Pearson:** misst gerade Linien. **Spearman:** misst Ränge, also jede Kurve, die nur steigt oder nur fällt.
- **Mutual Information:** Wie viel weniger unsicher bin ich über den Preis, wenn ich das Feature kenne? Jede Form, aber ohne Vorzeichen, und immer nur ein Feature allein.
- **VIF** = 1 / (1 − R²): Wie gut lässt sich ein Feature aus allen anderen vorhersagen? Ab 5 hinschauen, ab 10 ein Problem, ∞ = exakt berechenbar.

## Die Handänderungen (`prepare()`, für Train und Test)

| # | Handänderung | Grund |
|:--|:--|:--|
| 1 | `sqft_basement` füllen: `sqft_living − sqft_above` | exakte Regel |
| 2 | `sale_year` und `sale_month` aus `date` | `date` als Text unbrauchbar, Saisoneffekt möglich |
| 3 | `age_at_sale = sale_year − yr_built`, mindestens 0 | Alter sagt mehr als das Baujahr |
| 4 | `renovated` = 1, wenn `0 < yr_renovated ≤ sale_year` | kein Leakage durch spätere Renovierungen |
| 5 | `has_basement` = 1, wenn `sqft_basement > 0` | 59 % ohne Keller |
| 6 | löschen: `id`, `house_id`, `id_details`, `date`, `sqft_above`, `yr_renovated` | IDs ohne Information, `above` doppelt, `yr_renovated` durch 4 ersetzt |

**Faustregel:** Was zeilenweise nach einer festen Regel geht und nichts aus den Daten lernt → `prepare()`. Was etwas lernt (Median, häufigster Wert, Mittelwert/Streuung, Kategorienliste) → **Pipeline**, damit es in jedem CV-Fold nur aus den Trainingsdaten gelernt wird.

**In der Pipeline (Schritt 6):** häufigster Wert für `waterfront` und `view`, `log1p` für `sqft_lot` und `sqft_lot15`, `StandardScaler`, One-Hot für `zipcode`.